# Кросс-валидация на реальных данных 

Полезные ссылки: 
- [Пайплайны в sklearn (документация)](https://scikit-learn.org/1.5/modules/generated/sklearn.pipeline.Pipeline.html)
- [Пайплайны в sklearn (youtube)](https://www.youtube.com/watch?v=jzKSAeJpC6s)
- [Кросс-валидация в sklearn (на русском)](https://scikit-learn.ru/stable/modules/cross_validation.html)

Дополнительные материалы:
- [Кросс-валидация (Яндекс.образование)](https://education.yandex.ru/handbook/ml/article/kross-validaciya)
- [Data Splitting Strategies](https://amueller.github.io/aml/04-model-evaluation/1-data-splitting-strategies.html)
- [Resampling Approaches](https://allmodelsarewrong.github.io/resampling.html)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import  tqdm

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error


## Подготовка данных

И снова здравствуйте, Малые Кармакулы

In [ ]:
df = pd.read_csv('EDA_demo_data/karmakuly_processed.csv', parse_dates=['Date'], index_col='Date')
df.head()

#df = df['2015':]

print ('df shape with nans: ', df.shape)
df = df.drop (['tg', 'td', 'psl'], axis=1)
df = df.dropna()
print ('df shape without nans: ', df.shape)
df.head()



## Порождение дополнительных признаков

In [ ]:
df['year_pos'] = np.cos (np.pi + 2 * np.pi * df.index.dayofyear/365)
df['day_pos']  = np.cos (np.pi + 2 * np.pi * df.index.hour/24)

fig, ax = plt.subplots (1,2, figsize=(10,5))

df['2015':]['year_pos'].plot(ax = ax[0], title='year_pos (annual cycle)')
df['2015-01-01':'2015-01-03']['day_pos'].plot(ax = ax[1], title='day_pos (diurnal cycle)')

## Обучение и проверка модели 

Для начала посмотрим на оценки, если обучать и проверять модель на одних и тех же данных

In [ ]:
def evaluate_model (model, X, y):
       y_pred = model.predict(X)

       plt.figure()

       plt.hexbin (y, y_pred, mincnt=1, gridsize=50)

       plt.axis('equal')
       ax = plt.gca()
       lims = [
              np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
              np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
              ]
       plt.plot(lims, lims, 'k-', alpha=0.75, zorder=0)
       plt.xlim(lims)
       plt.ylim(lims)
       plt.grid()
       plt.xlabel('Observed')
       plt.ylabel('Predicted')

       r = np.corrcoef(y_pred, y)[0,1]
       r2 = r2_score(y, y_pred)
       mse = mean_squared_error(y, y_pred)

       plt.title ('RMSE = %.2f, R2 = %.2f'%(np.sqrt (mse), r2))

target_var = 'ta'

y = df[target_var]
X = df.drop(target_var, axis=1)

model = Pipeline([
                ('scaler', StandardScaler()),
                ('regressor', LinearRegression())
                ])

model.fit(X, y)
evaluate_model (model, X, y)



## Кросс-валидация штатными средствами sklearn №1

Используем функцию [cross_val_score](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.cross_val_score.html#sklearn.model_selection.cross_val_score).

Выбор метрик качества (scoring) для этой функции:
- [по имени](https://scikit-learn.org/stable/modules/model_evaluation.html#scoring-string-names)
- создание объекта типа Scorer на основе функции расчета метрики качества с помощью функции [make_scorer](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.make_scorer.html#sklearn.metrics.make_scorer)

In [ ]:
from sklearn.model_selection import ShuffleSplit, KFold, GroupKFold


#cv = ShuffleSplit(n_splits=10, test_size=0.25) #, random_state=0)
#cv = KFold (n_splits=4)
cv = GroupKFold (n_splits=4)

scores = cross_val_score (model, X, y, groups=seasons, cv=cv, scoring=scorer) #'neg_root_mean_squared_error')

print (scores)
print ('score mean: ', scores.mean()) 
print ('score std: ', scores.std()) 

In [ ]:
from sklearn.metrics import make_scorer, root_mean_squared_error
scorer = make_scorer (root_mean_squared_error)

display(scorer)

In [ ]:
def month_to_season(month):
    if month in [12, 1, 2]:
        return 1  # Winter
    elif month in [3, 4, 5]:
        return 2  # Spring
    elif month in [6, 7, 8]:
        return 3  # Summer
    elif month in [9, 10, 11]:
        return 4  # Autumn
    else:
        raise ValueError("Month must be between 1 and 12")

seasons = X.index.month.map(month_to_season)

## Посмотрим подробнее, как устроено разбиение train/test

In [ ]:
#cv = BlockedKFold (n_splits=5, block_size = 10000)
#cv = ShuffleSplit(n_splits=10, test_size=0.25) #, random_state=0)
#cv = KFold (n_splits=10)
is_train4folds = pd.DataFrame(index = X.index)

for i, (train_index, test_index) in enumerate(cv.split(X, groups=seasons)):
    is_train = np.zeros(X.index.shape,)
    is_train[train_index] = 1     
    is_train4folds[i] = is_train 

In [ ]:

plt.figure(figsize=(10,4))
plt.pcolormesh(X.index, is_train4folds.columns, is_train4folds.T)
plt.ylabel('folds')

## Кросс-валидация штатными средствами sklearn №2

Теперь используем более продвинутую функцию [cross_validate](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.cross_validate.html)

In [ ]:
scores = cross_validate (model, X, y, cv=cv, scoring=['neg_root_mean_squared_error', 'r2'], return_train_score = True, return_estimator=True)

In [ ]:
display(scores)

In [ ]:
print ('score mean (train): ', -scores['train_neg_root_mean_squared_error'].mean()) 
print ('score mean (test): ',  -scores['test_neg_root_mean_squared_error'].mean()) 

print ('score std (train): ', scores['train_neg_root_mean_squared_error'].std()) 
print ('score std (test): ', scores['test_neg_root_mean_squared_error'].std()) 

In [ ]:
#cv = ShuffleSplit(n_splits=5, test_size=0.25, random_state=0)
#cv = KFold(n_splits=5)
cv = BlockedKFold (n_splits=5, block_size = 10000)

is_train4folds = pd.DataFrame(index = df.index)

for i, (train_index, test_index) in enumerate(cv.split(X)):
    is_train = np.zeros(df.index.shape)
    is_train[train_index] = 1     
    is_train4folds[i] = is_train 


## Создаем собственный генератор, разделяющий выборку на train/test  

См. подробнее про [генераторы в Python на примере скатерти-самобранки](https://habr.com/ru/articles/560300/)

### Пример генератора в Python

In [ ]:
def my_range (n):
    for i in range(n):
        yield i

x = my_range (10) #np.arange (10)
display(f'x is {x}')
display(f'type(x) is {type(x)}')

print (next(x))
print (next(x))

# for i in x:
#     print (i)



### Новый класс для кросс-валидации

In [ ]:
class BlockedKFold:
    def __init__(self, n_splits=3, block_size = 500):
        self.n_splits = n_splits
        self.block_size = block_size
        self.splits = None
    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits
    def split(self, X, y=None, groups=None):
        n_samples = len(X)

        fold_size = self.block_size // self.n_splits
        block_start_idx = np.arange(0, n_samples, self.block_size)
        
        for i in range(self.n_splits):
            test_idx = []
            for block_start in block_start_idx:
                start = block_start + i * fold_size
                end = np.min([n_samples-1, block_start + (i + 1) * fold_size])
                test_idx.extend(list(range(start, end)))
            train_idx = list(set(range(n_samples)) - set(test_idx))

            train_idx = np.array(train_idx)
            test_idx = np.array(test_idx)
            
            yield train_idx, test_idx